In [1]:
# !pip install selenium beautifulsoup4 webdriver-manager

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
import random
import csv
from datetime import datetime
import re

In [48]:
# detail test
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
detail_href = "https://place.map.kakao.com/26435197"
driver.get(detail_href)


In [ ]:
# 테그 가져오기


html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# 한줄평
tag = soup.select_one('.desc_boxintro')
simple_desc = tag.get_text(strip=True) if tag else ""

print("한줄평은 다음과 같다: ", simple_desc)

# 태그
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

tags = extract_tags(soup)
print("태그는 다음과 같다: ", tags)

# 영업시간
opening_hour = extract_info(soup, '.info_operation .txt_detail')
print("영업시간:", opening_hour)

# 가격 정보 가져오기
price_list = soup.select('.section_booking .list_booking > li')
print(f"총 {len(price_list)}개의 가격 정보 수집 완료")
prices = {}
for _, room in enumerate(price_list):
    room_type = extract_info(room, '.tit_item')
    price = extract_info(room, '.info_item2')
    if room_type is None or price is None:
        continue
    prices[room_type] = price
    
# 리뷰 갯수
number_of_reviews = soup.select('.info_num')[0].get_text(strip=True) if soup.select('.info_num') else None
print(number_of_reviews)

# 카테고리 모텔 호텔 종류
raw_text = soup.select('.info_cate')[0]
category = raw_text.get_text(strip=True) if raw_text else None
category = category.replace("장소 카테고리", "").strip() if category else None

print("카테고리는 다음과 같다: ", category)

time.sleep(3)


한줄평은 다음과 같다:  
태그는 다음과 같다:  ['#2인실', '#MT펜션', '#가족실', '#단체민박', '#단체펜션', '#물놀이', '#바베큐장', '#산책로', '#오션뷰', '#온돌룸']
영업시간: 24시간 영업
총 4개의 예약 정보 수집 완료
룸타입5호실 45,000원~
룸타입11호실 45,000원~
룸타입8호실 50,000원~
룸타입7호실 50,000원~
7개
카테고리는 다음과 같다:  펜션


In [ ]:
#  리뷰 가져오기 제일 최신 순으로 3개

# 최신순 정렬
def click_latest_sort(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, "a.link_sort")

    for btn in buttons:
        text = driver.execute_script(
            "return arguments[0].textContent;", btn
        ).strip()

        if "최신" in text:
            if btn.find_elements(By.CSS_SELECTOR, ".ico_chk"):
                return True

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(0.3)

            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)

            return True

    return False
        
# 더보기 버튼 누르기
def load_reviews_with_more(driver):
    try:
        more_buttons = driver.find_elements(By.CSS_SELECTOR, ".btn_more")
        for btn in more_buttons:
            if btn.is_displayed():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.8)
    except:
        pass 

def get_recent_reviews(driver, year=2025, max_count=3):
    driver.execute_script("location.hash = '#review'")
    driver.switch_to.window(driver.window_handles[-1])
    
    print("리뷰 페이지로 이동 중...", driver.current_url)
    
    # 로딩 대기
    time.sleep(4)

    click_latest_sort(driver)
    load_reviews_with_more(driver)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    review_list = soup.select('.list_review .inner_review') # list 가져오기

    print(f"총 {len(review_list)}개의 리뷰 정보 수집 완료")

    reviews = []
    review_count = 0
    for r in review_list:
        date = r.select('.txt_date')[0].text if r.select('.txt_date') else None
        review_date = datetime.strptime(date, "%Y.%m.%d.")

        if review_date.year == year and review_count < max_count:
            describe = r.select('.desc_review')[0].text if r.select('.desc_review') else None  
            if describe is not None:
                review_count += 1
                
                # simple cleaning
                describe = describe.replace('접기', '')
                describe = describe.replace('\n', ' ')
                describe = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?]', '', describe)

                reviews.append(describe)
            else:
                continue
        elif review_date.year > year:
            continue
        elif review_count >= max_count:
            break
        else:
            break
    # tab close
    time.sleep(2)
    return reviews

In [ ]:
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

def extract_info(soup, tag_name):
    tag = soup.select_one(tag_name)
    text = tag.get_text(strip=True) if tag else None
    return text

def extract_details(detail_href):
    info = {}
    
    # 메인 페이지
    main_window = driver.current_window_handle
    time.sleep(5)
    
    driver.execute_script("window.open(arguments[0]);", detail_href)
    driver.switch_to.window(driver.window_handles[-1])
    
    print("상세 페이지로 이동 중...", driver.current_url)
    
    # 로딩 대기
    time.sleep(15)
    
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # 한줄설명
    simple_desc = extract_info(soup, '.desc_boxintro')
    print("한줄설명: ", simple_desc)
    info['한줄설명'] = simple_desc
    
    # 리뷰 갯수
    number_of_reviews = extract_info(soup, '.info_num')
    print("리뷰 개수: ", number_of_reviews)
    info['리뷰 개수'] = number_of_reviews
    
    # 태그
    tags = extract_tags(soup)
    print("태그: ", tags)
    info['태그'] = tags
    
    # 가격 가져오기
    price_list = soup.select('.section_booking .list_booking > li')
    print(f"총 {len(price_list)}개의 가격 정보 수집 완료")
    prices = {}
    for _, room in enumerate(price_list):
        room_type = extract_info(room, '.tit_item')
        price = extract_info(room, '.info_item2').replace("~", "")
        if room_type is None or price is None:
            continue
        prices[room_type] = price
    print("가격: ", prices)
    info['가격'] = prices
    
    # 영업시간
    opening_hour = extract_info(soup, '.info_operation .txt_detail')
    print("영업시간:", opening_hour)
    info['영업시간'] = opening_hour
    
    # 리뷰
    reviews = []
    reviews = get_recent_reviews(driver)
    # 리뷰 없으면 해당 숙소 패스
    if reviews is None or len(reviews) != 3:
        driver.close()
        driver.switch_to.window(main_window)
        time.sleep(2)
        return None
    print("리뷰: ", reviews)
    info['리뷰'] = reviews

    # 다시 검색 탭으로 전환
    driver.close()
    driver.switch_to.window(main_window)
    time.sleep(2)
    return info


In [86]:
# for testing 1 page
# 2 methods needed

def crawling(accommodation_list):
    informations = []
    for _, accommodation in enumerate(accommodation_list):
            info = {}
        # 장소명
            name = extract_info(accommodation, '.head_item > .tit_name > .link_name') 
            print("장소명:", name)
            info['장소명'] = name
        # 장소명 없으면 pass
            if name is None: 
                    continue
        
        # 평점
            score = extract_info(accommodation, '.rating > .score > em')
            print("평점:", score)
            info['평점'] = score
        # 평점 없으면 pass
            if score is None:
                    continue 
        
        # 주소
            addr = extract_info(accommodation, '.addr > p')
            print("주소:", addr)
            info['주소'] = addr
        # 주소가 없거나, 부산으로 시작하지 않으면 pass
            if addr is None or not addr.startswith("부산"):
                    continue

        # 구분
            info['구분'] = "숙박업체"
        
        # 카테고리
            category = extract_info(accommodation, '.subcategory')
            print("카테고리:", category)
            info['카테고리'] = category
        
        # 연락처
            phone_number = extract_info(accommodation, '.phone')
            print("연락처:", phone_number)
            info['연락처'] = phone_number

            a_tag = accommodation.select_one('.moreview')
            detail_href = a_tag.get('href') if a_tag else None
            details = extract_details(detail_href)
            if details is None:
                    continue
        
            info.update(details)
            info['출처'] = "카카오맵(2025)"
            print()
            print("--------------------------------------------------------------------------------------------------------")
            print("all the info: ", info)
            print("--------------------------------------------------------------------------------------------------------")
            informations.append(info)
            
    return informations

        

In [87]:
# kakao map crawling
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
url = "https://map.kakao.com/"
driver.get(url)

searchloc = '부산 숙박'

search_area = driver.find_element(
    By.CSS_SELECTOR,
    "input[type='text']"
)
search_area.send_keys(searchloc)  # 검색어 전달
search_area.send_keys(Keys.ENTER)

time.sleep(5)

driver.find_element(By.XPATH, r'//*[@id="info.main.options"]/li[2]/a').send_keys(Keys.ENTER) # 장소 탭 이동

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=143.0.7499.194)
Stacktrace:
0   chromedriver                        0x0000000104ef7dfc cxxbridge1$str$ptr + 3031016
1   chromedriver                        0x0000000104eefcb8 cxxbridge1$str$ptr + 2997924
2   chromedriver                        0x00000001049eab90 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74192
3   chromedriver                        0x00000001049c4948 chromedriver + 149832
4   chromedriver                        0x0000000104a59b38 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 528760
5   chromedriver                        0x0000000104a7258c _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 629708
6   chromedriver                        0x0000000104a2622c _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 317548
7   chromedriver                        

In [84]:
import pandas as pd
import os
# 저장할 파일명
filename = "busan_accommodation_result.csv"

page = 1   # 현재 전체 페이지 카운트
page2 = 1  # 1~5번 버튼 중 몇 번째인지

for i in range(1, 35):
    try:
        print(f"\n=== [ {page} 페이지 수집 시작 ] ===")
        
        btn_index = page2 # 현재 그룹(1~5) 내에서의 인덱스

        xpath = f'/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a[{btn_index}]'
        
        # 해당 페이지 버튼 클릭
        try:
            driver.find_element(By.XPATH, xpath).send_keys(Keys.ENTER)
        except:
            pass
            
        time.sleep(2) 
        
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        accommodation_list = soup.select('.placelist > .PlaceItem')
        print(f"찾은 숙소 개수: {len(accommodation_list)}개")
        
        if len(accommodation_list) == 0:
            print("데이터가 없습니다. (혹시 로딩 에러?)")
            continue

        # 크롤링 함수 실행
        informations = crawling(accommodation_list)
        print(f"수집 완료: {len(informations)} 건")

        # ---------------------------------------------------------
        # 2. CSV로 바로바로 저장 (Append Mode)
        # ---------------------------------------------------------
        df = pd.DataFrame(informations)
        
        # 파일이 없으면(첫 페이지면) -> 헤더 포함해서 새로 만들기 ('w' 모드)
        if not os.path.exists(filename):
            df.to_csv(filename, index=False, encoding='utf-8-sig', mode='w')
        # 파일이 있으면(2페이지부터) -> 헤더 빼고 이어 붙이기 ('a' 모드)
        else:
            df.to_csv(filename, index=False, encoding='utf-8-sig', mode='a', header=False)
            
        print(f"-> '{filename}'에 저장 완료!")
        
        # ---------------------------------------------------------
        # 3. 다음 페이지 그룹(Next 버튼) 이동 로직
        # ---------------------------------------------------------
        page += 1
        page2 += 1
        
        # page2가 5를 넘어가면 (즉, 6번째가 되면) -> '다음' 화살표 클릭하고 page2 초기화
        if page2 > 5:
            page2 = 1
            print(">> '다음' 버튼 클릭 (페이지 그룹 이동)")
            next_btn = driver.find_element(By.XPATH, r'//* [@id="info.search.page.next"]')
            
            # 다음 버튼이 비활성화(disabled) 상태면 종료
            if next_btn.get_attribute("disabled"):
                print("더 이상 다음 페이지가 없습니다. 종료합니다.")
                break
                
            next_btn.send_keys(Keys.ENTER)
            time.sleep(2) # 다음 묶음 로딩 대기

    except Exception as e:
        print(f"에러 발생 (중단): {e}")
        break
        
print('\n크롤링 전체 완료!')


=== [ 1 페이지 수집 시작 ] ===
찾은 숙소 개수: 15개
장소명: 롯데호텔 부산
평점: 4.3
주소: 부산 부산진구 가야대로 772
카테고리: 호텔
연락처: 051-810-1000
main window B2D20BD662DD6E95FC99D417A0A925EC
상세 페이지로 이동 중... https://place.map.kakao.com/7862727
한줄설명:  None
리뷰 개수:  116개
태그:  ['#2인실', '#가족실', '#물놀이', '#비즈니스호텔', '#사우나', '#실외수영장', '#연회장', '#이동약자접근', '#조식서비스', '#호캉스']
총 4개의 가격 정보 수집 완료
가격:  {'룸타입슈페리어 더블 (Superior Double)': '205,700원', '룸타입슈페리어 트윈 (Superior twin)': '205,700원', '룸타입디럭스 더블 (Deluxe Double)': '229,900원', '룸타입디럭스 트윈 (Deluxe Twin)': '229,900원'}
영업시간: 24시간 영업
리뷰 페이지로 이동 중... https://place.map.kakao.com/7862727#review
총 20개의 리뷰 정보 수집 완료
리뷰:  ['롯데호텔 알바한사람입니다  저도 일을 잘 한 편은 아니지만 , 정말 잘못된 체계라고 생각합니다 . 정직원을 뽑으시던지 계약직을 쓰시던지 알바를 시키면서 계약직을 원하시면 어떻게하자는건지 모르겠습니다 일을 알려주지도 않고 알아서 다 하기를 바라는 신기한 상황을 처음보아 후기를 남깁니다.  알바생을 진짜 직원인것처럼 부려 먹는 신기한 연회장,호텔입니다  ', '연회했는데 직원들이 센스가 하나도 없어서, 쓰레기통 가져다달라, 뜨거운물 가져다달라, 하나하나 말을 해야 겨우 챙겨주고, 서비스는 별1개도 아깝습니다. ', '서면 중심부에 있어서 교통도 편리하고 근처에 맛집도 많아서 좋아요. 호텔은 오래되었지만 관리가 잘되어 있어서 괜찮았어요. 저녁에 실외 수영장도 라이트업 되어서 예뻤고 사진

In [ ]:
page = 1  # 현재 페이지
page2 = 1  # 5개 중 몇번째인지(버튼이 5개씩있어서 6번째가 되면 다시 1로 바꿔줘야함)

filename = "busan_accommodation_result.csv" # 저장할 파일 명

for i in range(1, 34):
    try:
        print(f"\n=== [ {page} 페이지 수집 시작 ] ===")
        page2 += 1
        
        # 페이지 버튼 번호(1에서 5 사이 값)
        if i > 5:
            xpath = '/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a['+str(i-5)+']'
        else:
            xpath = '/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a['+str(i)+']'
        
        driver.find_element(By.XPATH, xpath).send_keys(Keys.ENTER)  # 페이지 선택
        
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser') # 모든 정보 가져오기
        
        accommodation_list = soup.select('.placelist > .PlaceItem')
        print(f"총 {len(accommodation_list)}개의 숙소 정보 수집 완료")
        print("크롤링 시작...")
        print()
        informations = crawling(accommodation_list)
        print(f"총 {len(informations)} 개의 정보를 읽어왔습니다.")
        
        # page2가 5를 넘어가면 다시 1로 바꿔주고 다음 버튼 클릭
        if page2 > 5:
            page2 = 1
            driver.find_element(By.XPATH, r'//* [@id="info.search.page.next"]').send_keys(Keys.ENTER)
        
        page += 1
    
    except:
        break
        
print('크롤링 완료')

1 page
크롤링 완료
